# M1 in simulation — pick one brick out of a flat lego pile

A UR3e with a Robotiq 2F-85 and a wrist RealSense, a plywood table, and a dozen real lego parts lying
flat on it. This notebook is the orchestrator: it stands the cell up and runs the two submodules in
order. Neither of them knows it is in a simulation.

| | |
|---|---|
| [`world.py`](world.py) | the cell — arm, gripper, camera, table, pile. Hardware, no decisions |
| [`submodule_1.py`](submodule_1.py) | look at the pile from two viewpoints, choose a brick, triangulate it, stand over it |
| [`submodule_2.py`](submodule_2.py) | descend, close, verify, lift, verify again |

**The perception is the real thing.** The rendered colour and depth go into
`m1.physical.submodule_3.analyse_pile` — the same function, thresholds and scoring that run against the
RealSense on the bench. What the simulator adds is a camera pose that is exactly right and a ground
truth to mark the answer against, which is the one thing the bench cannot provide.

Run the cells in order. Meshcat opens in a browser tab and animates as the arm moves.

In [ ]:
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pydrake.geometry import Meshcat

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from m1.physical import submodule_3 as perception
from m1.simulation import submodule_1, submodule_2
from m1.simulation import world as W


def show(image_bgr, title="", width=13):
    """Draw an OpenCV (BGR) image inline."""
    height = width * image_bgr.shape[0] / image_bgr.shape[1]
    plt.figure(figsize=(width, height))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

ImportError: cannot import name 'Meshcat' from 'pydrake' (/home/chaseungjoon/miniconda3/envs/airo-mono/lib/python3.10/site-packages/pydrake/__init__.py)

## 1. Stand up the cell

`Meshcat()` starts a viewer; open the URL it prints in a browser tab and leave it open — everything
below animates there. `build_world` then assembles the arm, bolts the gripper and camera to the
flange, lays the pile out flat and lets it settle.

Change `seed` for a different arrangement of the same parts, or pass `parts=[...]` for different ones
(anything in `lego_3d/urdf/`).

In [ ]:
meshcat = Meshcat()
print(f"Open the viewer at: {meshcat.web_url()}")

In [ ]:
world = W.build_world(meshcat, seed=3)

print(f"{len(world.bricks)} parts on the table:")
for brick, pose in world.brick_poses():
    x, y, z = pose.translation()
    print(f"  {brick.part:34s} at ({x:+.3f}, {y:+.3f}) m, {z * 1000:5.1f} mm up")

## 2. submodule_1 — find a brick and stand over it

Two viewpoints about 30 cm apart across the pile. Each one is perceived on its own; the brick chosen
is the one both views find, in the same place, with the best mean score. Its position then comes from
triangulating the two lines of sight through its centre, cross-checked against projecting each ray
onto the table plane raised by one brick height.

This is where the physical pipeline's submodule_3 (*which* brick) and submodule_1 (*where* it is, and
go there) become one step.

In [ ]:
target, views = submodule_1.run(world)
print()
print("Chosen brick:", target.describe())

### What each view saw

Every brick the perception found is outlined; the five it would send the arm at are numbered in
priority order with the jaw direction drawn across them. The same picture the physical pipeline writes
to `--debug-dir` on the bench.

In [ ]:
for view in views:
    show(perception.render_overlay(view.analysis), view.name)

In [ ]:
# The stage-by-stage panel for the first view: height above the table, the foreground the hysteresis
# admitted, the instance labels, and the regions that were dropped with the reason why.
show(perception.render_debug_panel(views[0].analysis), f"{views[0].name} — stages", width=8)

### Marking the homework

Nothing above consulted the simulator's ground truth. Now we can: how far is the perceived brick from
the real one, and is it even the right part?

In [ ]:
truth = submodule_1.score_against_truth(world, target)

print("how the position was arrived at")
print(f"  triangulated       {np.round(target.triangulated, 4)} m")
print(f"  ray-plane          {np.round(target.plane_projected, 4)} m")
print(f"  lines of sight miss each other by {target.triangulation_gap * 1000:.2f} mm")
print(f"  the two methods differ across the table by {target.method_disagreement * 1000:.2f} mm")
print(f"  used: {target.position_source}")
print()
print("against ground truth")
for key in ("part", "position_error_mm", "height_error_mm", "heading_error_deg"):
    print(f"  {key:20s} {truth[key]}")
print(f"  footprint            {truth['measured_footprint_mm']} mm measured vs {truth['true_footprint_mm']} mm real")

## 3. submodule_2 — grasp it and lift it

Descend the capped distance below the brick's top face, close to a width inside the brick so the pads
stall on it, check, lift, and check again after a pause. Both checks read the same signal the real
Robotiq gives: how far the fingers got compared to how far they were told to go.

In [ ]:
result = submodule_2.run(world, target)
print()
print(result.describe())

In [ ]:
if result.success:
    submodule_2.place(world, target)
submodule_2.park(world)

## 4. The whole cycle, several times over

Each pick disturbs the pile, so every round starts by looking at it again — which is the point: the
arrangement submodule_1 is choosing from is never the one it saw last time.

This is Module 1's loop. Deciding which bin each brick belongs in is Module 3's job; here they all go
to the same corner of the table so the next look at the pile is not confused by the last brick still
being in the gripper.

In [ ]:
def pick_one(world, avoid):
    """One full cycle. Returns (target, result), or (None, None) if there was nothing to grasp.

    ``avoid`` carries the bricks earlier rounds failed on. Without it the loop deadlocks: a failed
    grasp leaves the pile exactly as it was, so the same brick scores best again, forever.
    """
    try:
        target, _ = submodule_1.run(world, avoid=avoid)
    except RuntimeError as exception:
        print(f"submodule_1 stopped: {exception}")
        return None, None
    result = submodule_2.run(world, target)
    if result.success:
        submodule_2.place(world, target)
    else:
        avoid.append(target.position[:2])
    return target, result


picks, avoid = [], []
for round_number in range(5):
    print(f"\n{'=' * 78}\nround {round_number + 1}\n{'=' * 78}")
    target, result = pick_one(world, avoid)
    if result is None:
        break
    picks.append((target, result, submodule_1.score_against_truth(world, target)))

submodule_2.park(world)

In [ ]:
print(f"{'part':30s} {'grasped':>8s} {'pos err':>8s} {'width':>14s}  reason")
for target, result, truth in picks:
    print(
        f"{str(result.part):30s} {'yes' if result.success else 'NO':>8s} "
        f"{truth.get('position_error_mm', float('nan')):7.2f}mm "
        f"{target.width * 1000:5.1f}->{result.width_after_lift * 1000:5.1f}mm  {result.reason}"
    )

successes = sum(1 for _, result, _ in picks if result.success)
print(f"\n{successes}/{len(picks)} picked, {world.elapsed:.0f} s of simulated time.")

## Knobs worth turning

| where | what |
|---|---|
| `W.DEFAULT_PILE_PARTS`, `build_world(seed=...)` | which parts are in the pile and how they land |
| `W.PILE_CENTER`, `W.PILE_RADIUS` | how tightly packed the pile is — tighter means less fingertip room, which is what the grasp scoring is about |
| `W.RGB_NOISE_COUNTS`, `W.DEPTH_NOISE_M`, `W.DEPTH_DROPOUT_FRACTION` | how good the camera is. Turning the depth noise up is the honest way to see how much of the pipeline is leaning on it |
| `submodule_1.VIEWPOINTS` | where the two looks are taken from. Move them closer together and watch the triangulation gap grow |
| `perception.SCORE_WEIGHTS`, `perception.PRIORITY_MIN_CONFIDENCE` | what makes one brick a better grasp than another |
| `submodule_2.GRASP_DEPTH_M`, `GRIPPER_SQUEEZE_M` | how far down the brick's side the pads go, and how hard they pinch |